In [5]:
import pandas as pd 

nav_history = pd.read_csv("data/raw/02_nav_history.csv")
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [8]:
nav_history['date'] = pd.to_datetime(nav_history['date'])

print(nav_history.dtypes)

amfi_code             int64
date         datetime64[us]
nav                 float64
dtype: object


In [9]:
nav_history = nav_history.sort_values(by=["amfi_code", "date"]).reset_index(drop=True)
nav_history.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [13]:
nav_history["nav"] = nav_history.groupby("amfi_code")["nav"].ffill()

print("missing NAV values : ", nav_history["nav"].isnull().sum())

missing NAV values :  0


In [16]:
nav_history = nav_history[nav_history["nav"] > 0]

print("Invalid NAV values : ", (nav_history["nav"] <= 0).sum())

Invalid NAV values :  0


In [18]:
nav_history.to_csv("data/processed/clean_nav.csv", index=False)

print("clean_nav.csv saved successfully!")

clean_nav.csv saved successfully!


In [21]:
import pandas as pd
import os

transactions = pd.read_csv("data/raw/08_investor_transactions.csv")

print(transactions.head())
print(transactions.columns.tolist())

  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verified  
2      M

In [22]:
nav_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[us]
 2   nav        46000 non-null  float64       
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 1.1 MB


In [28]:
print(transactions.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [27]:
transactions["transaction_type"] = (transactions["transaction_type"]
                                    .str.strip()
                                    .str.title()
                                    )

valid_types = ["Sip", "Lumpsum", "redemption"]

transactions = transactions[transactions["transaction_type"].isin(valid_types)]

transactions = transactions[transactions["amount_inr"] > 0]

transactions["kyc_status"] = (
    transactions["kyc_status"]
    .str.strip()
    .str.upper()
)

transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"],
                                                  errors="coerce")

transactions = transactions.dropna(subset=["transaction_date"]) 

os.makedirs("data/processes", exist_ok=True)

transactions.to_csv("data/processed/clean_transactions.csv", index=False)

print("clean_transactions.csv saved successfully!")

clean_transactions.csv saved successfully!


In [1]:
import pandas as pd
import numpy as np
import os

performance = pd.read_csv("data/raw/07_scheme_performance.csv")

print(performance.head())
print(performance.columns.tolist())

   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \
0              11.49

In [3]:
print(performance.columns.tolist())

['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [6]:
performance["return_1yr_pct"] = pd.to_numeric(performance["return_1yr_pct"], errors="coerce")
performance["return_3yr_pct"] = pd.to_numeric(performance["return_3yr_pct"], errors="coerce")
performance["return_5yr_pct"] = pd.to_numeric(performance["return_5yr_pct"], errors="coerce")

performance["negative_sharpe"] = performance["sharpe_ratio"] < 0

performance = performance[
    (performance["expense_ratio_pct"] >= 0.1)&
    (performance["expense_ratio_pct"] <= 2.5)
]

performance = performance.dropna(subset=["return_1yr_pct", "return_3yr_pct", "return_5yr_pct"])

os.makedirs("data/processed", exist_ok=True)
performance.to_csv("data/processed/clean_performance.csv", index=False)

print("clean_performance.csv saved sucessfully!")

clean_performance.csv saved sucessfully!
